Sequoia package for kinship reconstruction

In [7]:
#Make sure R can find packages installed through Conda
.libPaths(c('/home/jbos/.conda/envs/vcfR','/home/jbos/.conda/envs/vcfR/lib/R/library'))

In [8]:
library(sequoia)
library(vcfR)
library(sf)

In [3]:
#Function to make input for Sequoia, despite the fact that our samples don't have sex or birth year
makeLifeHistory<-function(vcf){
    lh<-as.data.frame(colnames(vcf@gt)[2:length(colnames(vcf@gt))])
    lh$Sex<-4
    lh$BirthYear<--1
    colnames(lh)<-c('ID','Sex','BirthYear')
    return(lh)
    }

In [4]:
#Calculate great circle distances from lat/lons
wide_dists <- function(sites){
    x<-st_as_sf(sites,coords=c('LONGITUDE','LATITUDE'))
    x<-st_set_crs(x,4326)
    d <- st_distance(x)
    colnames(d)<-sites$Name
    rownames(d)<-sites$Name
    d<-as.data.frame(d)
    return(d)
}

In [9]:
#Check distance between potential relatives
checkDist<-function(mayberel,dists,PairNumber){
    pn<-PairNumber
    p<-unlist(strsplit(mayberel$MaybePar$ID1[pn],"_"))[1]
    o<-unlist(strsplit(mayberel$MaybePar$ID2[pn],"_"))[1]
    return(dists[p,o])
    }

In [11]:
#Read in SNP data for each taxon
vcf1<-read.vcfR("/scratch/jbos/spp1/pruned_snps10.vcf")
vcf2<-read.vcfR("/scratch/jbos/spp2/pruned_snps10.vcf")
vcf3<-read.vcfR("/scratch/jbos/spp3/pruned_snps10.vcf")
vcf4<-read.vcfR("/scratch/jbos/spp4/pruned_snps10.vcf")

Scanning file to determine attributes.
File attributes:
  meta lines: 361
  header_line: 362
  variant count: 2292
  column count: 90
Meta line 361 read in.
All meta lines processed.
gt matrix initialized.
Character matrix gt created.
  Character matrix gt rows: 2292
  Character matrix gt cols: 90
  skip: 0
  nrows: 2292
  row_num: 0
Processed variant: 2292
All variants processed
Scanning file to determine attributes.
File attributes:
  meta lines: 339
  header_line: 340
  variant count: 1678
  column count: 63
Meta line 339 read in.
All meta lines processed.
gt matrix initialized.
Character matrix gt created.
  Character matrix gt rows: 1678
  Character matrix gt cols: 63
  skip: 0
  nrows: 1678
  row_num: 0
Processed variant: 1678
All variants processed
Scanning file to determine attributes.
File attributes:
  meta lines: 313
  header_line: 314
  variant count: 1213
  column count: 50
Meta line 313 read in.
All meta lines processed.
gt matrix initialized.
Character matrix gt created.

In [12]:
#Make mostly empty life history inputs for Sequoia functions
lh1<-makeLifeHistory(vcf1)
lh2<-makeLifeHistory(vcf2)
lh3<-makeLifeHistory(vcf3)
lh4<-makeLifeHistory(vcf4)

In [14]:
sequoia1<-GenoConvert(InFile="/scratch/jbos/spp1/pruned_snps_raw10.raw",OutFile='snps1_miss.txt',OutFormat='seq')
sequoia2<-GenoConvert(InFile="/scratch/jbos/spp2/pruned_snps_raw10.raw",OutFile='snps2_miss.txt',OutFormat='seq')
sequoia3<-GenoConvert(InFile="/scratch/jbos/spp3/pruned_snps_raw10.raw",OutFile='snps3_miss.txt',OutFormat='seq')
sequoia4<-GenoConvert(InFile="/scratch/jbos/spp4/pruned_snps_raw10.raw",OutFile='snps4_miss.txt',OutFormat='seq')

Warning message:
“ There are 15 monomorphic (fixed) SNPs, these will be excluded 
”
After exclusion, There are  81  individuals and  2277  SNPs.


Warning message:
“ There are 8 monomorphic (fixed) SNPs, these will be excluded 
”
After exclusion, There are  54  individuals and  1670  SNPs.


Warning message:
“ There are 7 monomorphic (fixed) SNPs, these will be excluded 
”
After exclusion, There are  41  individuals and  1206  SNPs.


Warning message:
“ There are 4 monomorphic (fixed) SNPs, these will be excluded 
”
After exclusion, There are  39  individuals and  1192  SNPs.




In [15]:
#Make genome inputs
sequoia1<-as.matrix(read.table("snps1_miss.txt", row.names=1, header=FALSE))
sequoia2<-as.matrix(read.table("snps2_miss.txt", row.names=1, header=FALSE))
sequoia3<-as.matrix(read.table("snps3_miss.txt", row.names=1, header=FALSE))
sequoia4<-as.matrix(read.table("snps4_miss.txt", row.names=1, header=FALSE))

In [16]:
#Find potential relatives with no age prior
MaybeRel1 <- GetMaybeRel(GenoM = sequoia1, Err = 0.005, LifeHistData = lh1, quiet=FALSE,Herm='A')
MaybeRel1

Searching for non-assigned parent-offspring pairs ... (Module = par)

Warning message:
“ There are 15 monomorphic (fixed) SNPs, these will be excluded 
”
After exclusion, There are  81  individuals and  2277  SNPs.


Ageprior: Flat 0/1, overlapping generations, MaxAgeParent = 99,99



Counting opposing homozygous loci between all individuals ...
Checking for non-assigned Parent-Offspring pairs ...


Found 1 likely parent-offspring pairs, and 0 other non-assigned pairs of possible relatives



,ID1,ID2,TopRel,LLR,OH,BirthYear1,BirthYear2,AgeDif,Sex1,Sex2,SNPdBoth
,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,LEY11_T15,LEY04_T15,PO,13.46,0,NA,NA,NA,4,4,2254


In [17]:
#Find potential relatives with no age prior
MaybeRel2 <- GetMaybeRel(GenoM = sequoia2, Err = 0.005, LifeHistData = lh2, quiet=FALSE,Herm='A')
MaybeRel2

Searching for non-assigned parent-offspring pairs ... (Module = par)

Warning message:
“ There are 8 monomorphic (fixed) SNPs, these will be excluded 
”
After exclusion, There are  54  individuals and  1670  SNPs.


Ageprior: Flat 0/1, overlapping generations, MaxAgeParent = 99,99



Counting opposing homozygous loci between all individuals ...
Checking for non-assigned Parent-Offspring pairs ...


Found 1 likely parent-offspring pairs, and 0 other non-assigned pairs of possible relatives



,ID1,ID2,TopRel,LLR,OH,BirthYear1,BirthYear2,AgeDif,Sex1,Sex2,SNPdBoth
,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,LEY14_T09,CEB07_T14,PO,25.35,1,NA,NA,NA,4,4,1630


In [18]:
#Find potential relatives with no age prior
MaybeRel3 <- GetMaybeRel(GenoM = sequoia3, Err = 0.005, LifeHistData = lh3, quiet=FALSE,Herm='A',Tassign=3)
MaybeRel3

Searching for non-assigned parent-offspring pairs ... (Module = par)

Warning message:
“ There are 7 monomorphic (fixed) SNPs, these will be excluded 
”
After exclusion, There are  41  individuals and  1206  SNPs.


Ageprior: Flat 0/1, overlapping generations, MaxAgeParent = 99,99



Counting opposing homozygous loci between all individuals ...
Checking for non-assigned Parent-Offspring pairs ...


Found 1 likely parent-offspring pairs, and 0 other non-assigned pairs of possible relatives



,ID1,ID2,TopRel,LLR,OH,BirthYear1,BirthYear2,AgeDif,Sex1,Sex2,SNPdBoth
,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,CEB26_T22,CEB05_T12,PO,11.73,0,NA,NA,NA,4,4,1185


In [19]:
#Find potential relatives with no age prior
MaybeRel4 <- GetMaybeRel(GenoM = sequoia4, Err = 0.005, LifeHistData = lh4, quiet=FALSE,Herm='A',Tassign=3)
MaybeRel4

Searching for non-assigned parent-offspring pairs ... (Module = par)

Warning message:
“ There are 4 monomorphic (fixed) SNPs, these will be excluded 
”
After exclusion, There are  39  individuals and  1192  SNPs.


Ageprior: Flat 0/1, overlapping generations, MaxAgeParent = 99,99



Counting opposing homozygous loci between all individuals ...
Checking for non-assigned Parent-Offspring pairs ...
Checking for Parent-Parent-Offspring trios ...


Found 3 likely parent-offspring pairs, and 0 other non-assigned pairs of possible relatives



,ID1,ID2,TopRel,LLR,OH,BirthYear1,BirthYear2,AgeDif,Sex1,Sex2,SNPdBoth
,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,CEB01_T12,CEB23_T16,PO,4.77,0,NA,NA,NA,4,4,1056
2,CEB01_T11,CEB23_T16,PO,4.22,0,NA,NA,NA,4,4,1054
3,CEB01_T11,CEB01_T12,PO,3.81,0,NA,NA,NA,4,4,1139


In [22]:
#Import site locations with lat/lons
sites <- read.csv("/home/jbos/Atenuis_Philippines/metadat/all_Atenuis_sites.csv")

In [23]:
#Create distance matrix from sites
dists<-wide_dists(sites)

In [27]:
#Check distance between relatives and convert to km, taxa 1
checkDist(MaybeRel1,dists,1)/1000
checkDist(MaybeRel1,dists,2)/1000

58.33163 [m]

numeric(0)

In [26]:
#Check distance between relatives and convert to km, taxa 2
checkDist(MaybeRel2,dists,1)/1000

142.4513 [m]

In [25]:
#Check distance between relatives and convert to km, taxa 3
checkDist(MaybeRel3,dists,1)/1000

174.1064 [m]

In [24]:
#Check distance between relatives and convert to km, taxa 4
checkDist(MaybeRel4,dists,1)/1000
checkDist(MaybeRel4,dists,2)/1000
checkDist(MaybeRel4,dists,3)/1000

180.5587 [m]

180.5587 [m]

0 [m]